In [1]:
import pandas as pd
from rdkit import Chem

In [10]:
with Chem.SDMolSupplier('enamine_crbn_database/enamine_KNIME_processed_dataset.sdf') as suppl:
    molecules = [mol for mol in suppl]
data_dict = {}
for mol in molecules:
    smiles = mol.GetProp('Molecule')
    label = mol.GetProp('Mode of Action')
    data_dict[smiles] = label
df_large_cov = pd.DataFrame.from_dict(data_dict, orient='index', columns=['label'])
df_large_cov.reset_index(inplace=True)
df_large_cov = df_large_cov.rename(columns={'index': 'smiles'})
df_large_cov.loc[df_large_cov['label'] == 'Covalent', 'cov_label'] = int(1)
df_large_cov.loc[df_large_cov['label'] == 'Noncovalent', 'cov_label'] = int(0)
#
with Chem.SDMolSupplier('open_crbn_database/bindingDB_mgDB_KNIME_processed_dataset.sdf') as suppl:
    molecules = [mol for mol in suppl]
data_dict = {}
for mol in molecules:
    smiles = mol.GetProp('Molecule (Canonical)')
    label = mol.GetProp('ModeOfAction')
    data_dict[smiles] = label
df_small_cov = pd.DataFrame.from_dict(data_dict, orient='index', columns=['label'])
df_small_cov.reset_index(inplace=True)
df_small_cov = df_small_cov.rename(columns={'index': 'smiles'})
df_small_cov.loc[df_small_cov['label'] == 'covalent', 'cov_label'] = int(1)
df_small_cov.loc[df_small_cov['label'] == 'non-covalent', 'cov_label'] = int(0)
df_large_cov = pd.concat([df_large_cov, df_small_cov], ignore_index=True, axis=0).reset_index(drop=True)
df_large_cov['crbn_chemspace'] = int(1)
#
df_cov_enh = pd.read_csv('enamine_covalent_database/covalent_fragment_enhancement_MW250.csv')
df_cov_enh.loc[df_cov_enh['label'] == 'Covalent', 'cov_label'] = int(1)
df_cov_enh.loc[df_cov_enh['label'] != 'Covalent', 'cov_label'] = int(0)
df_cov_enh.rename(columns={'Molecule (Canonical)': 'smiles'}, inplace=True)
df_cov_enh.drop(columns=['ExactMW'], inplace=True)
df_cov_enh['crbn_chemspace'] = int(0)
df_large_cov = pd.concat([df_large_cov, df_cov_enh], ignore_index=True, axis=0).reset_index(drop=True)
#
df_large_cov.drop_duplicates(subset=['smiles'], keep='first', inplace=True)
df_large_cov.reset_index(drop=True, inplace=True)

In [11]:
df_large_cov

,smiles,label,cov_label,crbn_chemspace
0,O=C1CCC(N2Cc3ccc(C(=O)O)cc3C2=O)C(=O)N1,Noncovalent,0.0,1
1,O=C1CCC(N2Cc3cc(C(=O)O)ccc3C2=O)C(=O)N1,Noncovalent,0.0,1
2,CNC(=O)c1ccc2c(c1)CN(C1CCC(=O)NC1=O)C2=O,Noncovalent,0.0,1
3,CCNC(=O)c1ccc2c(c1)CN(C1CCC(=O)NC1=O)C2=O,Noncovalent,0.0,1
4,O=C1CCC(N2Cc3cc(C(=O)NC4CC4)ccc3C2=O)C(=O)N1,Noncovalent,0.0,1
...,...,...,...,...
8244,C=CS(=O)(=O)NCC(C)CN1CCOCC1,Covalent,1.0,0
8245,C=CS(=O)(=O)N(CC)C1CCOC(C)(C)C1,Covalent,1.0,0
8246,C=CS(=O)(=O)N[C@@H](C)COCC,Covalent,1.0,0
8247,C=CS(=O)(=O)NC(c1nccn1C)C(C)C,Covalent,1.0,0


In [12]:
df_large_cov['cov_label'].value_counts()

cov_label
0.0    4170
1.0    4079
Name: count, dtype: int64

In [13]:
df_large_cov['crbn_chemspace'].value_counts()

crbn_chemspace
1    4320
0    3929
Name: count, dtype: int64

In [14]:
df_large_cov.to_csv('combined_covalent_noncovalent_dataset.csv', index=False)